# Phase 1 — does the top eigenvalue predict universal vulnerability?

<a href="https://colab.research.google.com/github/fatemehsalmani75/active-subspace-bloodmnist/blob/main/notebooks/phase1_spectrum_diagnostic.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Hypothesis.** How vulnerable a classifier is to universal perturbations is set by the leading eigenvalues of
$C = \mathbb{E}[\nabla_x \ell \, \nabla_x \ell^T]$. To first order, a perturbation $v$ with $\|v\|_2 \le \delta$ changes the
loss by $g^T v$, and $\max_{\|v\| \le \delta} \mathbb{E}[(g^T v)^2] = \delta^2 \lambda_1$ (at $v = \delta u_1$).

**Prediction (written before running).** Across six differently trained models (base, noise, AS fine-tuning,
UAP fine-tuning, UAT, PGD-AT), the universal fooling rate rises with $\lambda_1$ and with $\lambda_1 / \operatorname{tr} C$.

**Decision rule.** Spearman $\rho \ge 0.8$ between a spectral measure and the AS fooling rate → hypothesis supported,
go to Phase 2. $\rho < 0.5$ for every measure → hypothesis rejected, skip Phase 2. In between → report which measure
predicts best and go to Phase 2.

Use a T4 GPU. About 75 min. Everything is saved in Google Drive (`active_subspace_runs`); after a disconnect press
*Run all* again and it continues where it stopped.

In [ ]:
%cd /content
!test -d active-subspace-bloodmnist || git clone -q https://github.com/fatemehsalmani75/active-subspace-bloodmnist.git
%cd /content/active-subspace-bloodmnist
!git pull -q
!pip install -q medmnist adversarial-robustness-toolbox

In [ ]:
import os, copy
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from asnet.data import NUM_CLASSES, get_datasets, make_loader, subset
from asnet.models import build_model
from asnet.utils import set_seed, get_device, train_classifier, stack_dataset
from asnet.attack import as_universal_attack, fooling_rate, predict
from asnet.uap_baseline import uap_baseline
from asnet.robust import (adv_finetune, uat_finetune, pgd_finetune,
                          as_perturbation, uap_perturbation, noise_perturbation)
from asnet.spectrum import input_spectrum

OUT = "/content/drive/MyDrive/active_subspace_runs"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive not available, saving in this runtime only:", e)
    OUT = "runs"
MODELS = os.path.join(OUT, "models")
os.makedirs(MODELS, exist_ok=True)
CSV = os.path.join(OUT, "phase1_spectrum.csv")
BASE_CKPT = os.path.join(OUT, "vgg19_bloodmnist.pt")   # the same base model as the defense benchmark

set_seed(0)
device = get_device()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(device)

## Settings (same as the defense benchmark)

In [ ]:
SIZE, EPOCHS, FT_EPOCHS = 64, 20, 5
DELTA, REFRESH, PGD_EPS = 4, 60, 2.0
N_ATTACK, EVAL_SEEDS = 200, [0, 1]
N_SPECTRUM = 2000          # clean training images for the spectrum (exact SVD)
MODEL_NAMES = ["none (base)", "noise", "AS fine-tuning", "UAP fine-tuning", "UAT", "PGD-AT"]

## Data and base model

In [ ]:
train_aug, _, _ = get_datasets(SIZE, augment=True)
train, val, test = get_datasets(SIZE)
train_loader = make_loader(train_aug, 128, shuffle=True)

base = build_model("vgg19", NUM_CLASSES, SIZE)
if os.path.exists(BASE_CKPT):
    print("loading the base model saved in", OUT)
    base.load_state_dict(torch.load(BASE_CKPT, map_location="cpu"))
else:
    base, _ = train_classifier(base, train_loader, make_loader(val, 256), epochs=EPOCHS, device=device)
    torch.save(base.state_dict(), BASE_CKPT)
base.to(device).eval()

Xte, yte = stack_dataset(test, device=device)
Xa, ya = stack_dataset(subset(train, 100, seed=300), device=device)
spec_loader = make_loader(subset(train, N_SPECTRUM, seed=10), 100)
v_shape = Xte[:1].shape

## Train (or load) the six models

Each trained model is saved to Drive and reused in Phase 2.

In [ ]:
def path(name):
    return os.path.join(MODELS, name.replace(" ", "_").replace("(", "").replace(")", "") + ".pt")

def train_one(name):
    set_seed(1)
    m = copy.deepcopy(base)
    if name == "noise":
        m, t = adv_finetune(m, train_loader, noise_perturbation(v_shape, DELTA), epochs=FT_EPOCHS, refresh=REFRESH, device=device)
    elif name == "AS fine-tuning":
        m, t = adv_finetune(m, train_loader, as_perturbation(Xa, ya, DELTA), epochs=FT_EPOCHS, refresh=REFRESH, device=device)
    elif name == "UAP fine-tuning":
        m, t = adv_finetune(m, train_loader, uap_perturbation(Xa, NUM_CLASSES, DELTA), epochs=FT_EPOCHS, refresh=REFRESH, device=device)
    elif name == "UAT":
        m, t = uat_finetune(m, train_loader, DELTA, epochs=FT_EPOCHS, device=device)
    elif name == "PGD-AT":
        m, t = pgd_finetune(m, train_loader, PGD_EPS, epochs=FT_EPOCHS, device=device)
    return m, t

models, times = {"none (base)": base}, {"none (base)": 0.0}
for name in MODEL_NAMES[1:]:
    m = build_model("vgg19", NUM_CLASSES, SIZE)
    if os.path.exists(path(name)):
        print(f"{name}: loading from Drive")
        ck = torch.load(path(name), map_location="cpu")
        m.load_state_dict(ck["state_dict"]); t = ck["time"]
    else:
        print(f"{name}: training")
        m, t = train_one(name)
        torch.save({"state_dict": m.state_dict(), "time": t}, path(name))
    models[name], times[name] = m.to(device).eval(), t

## Spectrum and universal fooling for each model

In [ ]:
results = pd.read_csv(CSV).to_dict("records") if os.path.exists(CSV) else []
done = {r["model"] for r in results}
spectra = {}

for name, m in models.items():
    for p in m.parameters():
        p.requires_grad_(False)
    s = input_spectrum(m, spec_loader, device=device)
    spectra[name] = s["eigvals"].numpy()
    if name in done:
        print(f"{name}: already evaluated"); continue
    row = {"model": name, "train time (s)": times[name],
           "clean acc": (predict(m, Xte) == yte).float().mean().item(),
           **{k: s[k] for k in ["lambda1", "trace", "lambda1_share", "top5_share", "gain_over_random"]}}
    fa, fu = [], []
    for seed in EVAL_SEEDS:
        Xs, ys = stack_dataset(subset(train, N_ATTACK, seed=500 + seed), device=device)
        fa.append(fooling_rate(m, Xte, as_universal_attack(m, Xs, ys, delta=DELTA, seed=seed, log=lambda *a: None).v))
        fu.append(fooling_rate(m, Xte, uap_baseline(m, Xs, NUM_CLASSES, delta=DELTA, device=device)))
    row.update({"AS fooling": np.mean(fa), "AS fooling std": np.std(fa, ddof=1),
                "UAP fooling": np.mean(fu), "UAP fooling std": np.std(fu, ddof=1)})
    results.append(row)
    pd.DataFrame(results).to_csv(CSV, index=False)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})

res = pd.DataFrame(results).set_index("model").loc[MODEL_NAMES]
res.round(4)

## The decision

In [ ]:
measures = ["lambda1", "lambda1_share", "top5_share", "trace"]
rows = []
for meas in measures:
    for target in ["AS fooling", "UAP fooling"]:
        rho, p = spearmanr(res[meas], res[target])
        rows.append({"measure": meas, "target": target, "spearman rho": round(rho, 3), "p": round(p, 3)})
corr = pd.DataFrame(rows)
display(corr.pivot(index="measure", columns="target", values="spearman rho"))

best = corr[corr.target == "AS fooling"].sort_values("spearman rho", ascending=False).iloc[0]
spectral = corr[(corr.target == "AS fooling") & (corr.measure != "trace")]
if spectral["spearman rho"].max() >= 0.8:
    verdict = f"SUPPORTED: {best.measure} predicts AS fooling with rho = {best['spearman rho']}. Go to Phase 2."
elif spectral["spearman rho"].max() < 0.5:
    verdict = "REJECTED: no spectral measure reaches rho 0.5. Skip Phase 2; the paper is the empirical analysis."
else:
    verdict = f"PARTLY SUPPORTED: best is {best.measure} (rho = {best['spearman rho']}). Report it and go to Phase 2."
print(verdict)
corr.to_csv(os.path.join(OUT, "phase1_correlations.csv"), index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
colors = dict(zip(MODEL_NAMES, ["#888", "#c9a227", "#3b6ea5", "#d9822b", "#8b5fbf", "#4a9b6e"]))

for name in MODEL_NAMES:
    lam = spectra[name][:100]
    axes[0].semilogy(np.arange(1, len(lam) + 1), lam, color=colors[name], label=name)
axes[0].set(xlabel="index i", ylabel=r"$\lambda_i$", title="input-gradient spectrum of each model")
axes[0].legend(fontsize=7)

for ax, meas, lab in [(axes[1], "lambda1", r"$\lambda_1$"), (axes[2], "lambda1_share", r"$\lambda_1 / \mathrm{tr}\, C$")]:
    for name in MODEL_NAMES:
        ax.errorbar(res.loc[name, meas], res.loc[name, "AS fooling"] * 100, yerr=res.loc[name, "AS fooling std"] * 100,
                    fmt="o", ms=8, color=colors[name], capsize=3, label=name)
    rho = corr[(corr.measure == meas) & (corr.target == "AS fooling")]["spearman rho"].iloc[0]
    ax.set(xlabel=lab, ylabel="AS universal fooling (%)", title=f"Spearman rho = {rho}")
    if meas == "lambda1":
        ax.set_xscale("log")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "phase1_spectrum.png"), bbox_inches="tight"); plt.show()